# R20 - Queda de Pressao em Tubo Rugoso: Grupos Adimensionais

## Objetivo
Determinar, pelo Teorema Pi de Buckingham, os grupos adimensionais necessarios para descrever a queda de pressao $p$ em um tubo com parede rugosa.

## Fundamentacao Teorica

**Variaveis** ($n=7$): $p,\ v,\ \rho,\ \mu,\ l,\ d,\ e$ (rugosidade)

| Variavel | Descricao | Dimensoes |
|---|---|---|
| $p$ | queda de pressao | $M L^{-1} T^{-2}$ |
| $v$ | velocidade | $L T^{-1}$ |
| $\rho$ | densidade | $M L^{-3}$ |
| $\mu$ | viscosidade | $M L^{-1} T^{-1}$ |
| $l$ | comprimento | $L$ |
| $d$ | diametro | $L$ |
| $e$ | rugosidade media | $L$ |

**Dimensoes** ($m=3$): $M, L, T$

**Grupos Pi**: $p = n - m = 7 - 3 = 4$ grupos.

Variaveis de repeticao: $v, \rho, d$. Os 4 grupos sao:

| Grupo | Expressao | Nome |
|---|---|---|
| $\Pi_1$ | $p/(\rho v^2)$ | Numero de Euler $Eu$ |
| $\Pi_2$ | $\rho v d/\mu$ | Numero de Reynolds $Re$ |
| $\Pi_3$ | $l/d$ | Razao comprimento/diametro |
| $\Pi_4$ | $e/d$ | Rugosidade relativa |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Propriedades da agua a 20C
rho = 998.2    # kg/m^3
mu  = 1.002e-3 # Pa.s

# Geometria do tubo
d = 0.05    # m (50 mm)
l = 10.0    # m
e = 0.046e-3  # m (rugosidade do aco comercial)

# Velocidades de fluxo
v_vec = np.linspace(0.1, 5.0, 200)

Re  = rho * v_vec * d / mu
eps_rel = e / d    # rugosidade relativa (constante para tubo fixo)
l_d = l / d        # razao l/d

print('=== Grupos adimensionais ===')
print(f'Rugosidade relativa  e/d  = {eps_rel:.5f}')
print(f'Razao geom.          l/d  = {l_d:.1f}')
print()
print('Numero de Reynolds para diferentes velocidades:')
for v in [0.5, 1.0, 2.0, 5.0]:
    Re_v = rho * v * d / mu
    regime = 'laminar' if Re_v < 2300 else ('transicao' if Re_v < 4000 else 'turbulento')
    print(f'  v={v:.1f} m/s -> Re={Re_v:.0f}  ({regime})')

In [ ]:
# Fator de Darcy-Weisbach considerando regime (Colebrook simplificado)
def fator_atrito(Re, eps_rel):
    if Re < 2300:  # laminar
        return 64.0 / Re
    elif Re < 4000:  # transicao - interpolacao
        f_lam = 64.0 / 2300
        f_tur = 0.316 * Re**(-0.25)  # Blasius
        t = (Re - 2300) / (4000 - 2300)
        return f_lam + t * (f_tur - f_lam)
    else:  # turbulento (Moody: Colebrook-White aprox.)
        return 0.316 * Re**(-0.25)

f_vec = np.array([fator_atrito(re, eps_rel) for re in Re])
Eu_vec = f_vec * (l/d) / 2   # Euler = f*(l/d)/2
dp_vec = Eu_vec * rho * v_vec**2  # queda de pressao real

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Diagrama de Moody simplificado
axes[0].loglog(Re, f_vec, lw=2.5, color='steelblue')
axes[0].axvline(2300, color='orange', ls='--', lw=1.5, label='Re=2300 (lam.)')
axes[0].axvline(4000, color='red', ls='--', lw=1.5, label='Re=4000 (turb.)')
axes[0].set_xlabel('Numero de Reynolds Re', fontsize=12)
axes[0].set_ylabel('Fator de atrito f', fontsize=12)
axes[0].set_title('Diagrama de Moody (simplificado)', fontsize=13)
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.4)

# Queda de pressao vs velocidade
axes[1].plot(v_vec, dp_vec/1000, lw=2.5, color='darkorange')
axes[1].set_xlabel('Velocidade v (m/s)', fontsize=12)
axes[1].set_ylabel('Queda de pressao p (kPa)', fontsize=12)
axes[1].set_title(f'Dp vs v  (l={l}m, d={d*1000:.0f}mm, e/d={eps_rel:.4f})', fontsize=12)
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('r20_tubo_rugoso.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tabela comparativa de rugosidades
materiais = {
    'Material'      : ['Plastico/vidro', 'Aco comercial', 'Ferro fundido', 'Concreto'],
    'e (mm)'        : [0.0015, 0.046, 0.26, 1.0],
}
df = pd.DataFrame(materiais)
df['e/d (d=50mm)'] = (df['e (mm)'] * 1e-3 / d).round(6)
df['Regime Re=50000'] = ['turbulento liso', 'rugoso transicao', 'rugoso', 'muito rugoso']
print(df.to_string(index=False))

## Conclusao

Com $n=7$ variaveis e $m=3$ dimensoes, obtem-se $p = 4$ grupos adimensionais:

$$Eu = \frac{p}{\rho v^2}, \quad Re = \frac{\rho v d}{\mu}, \quad \frac{l}{d}, \quad \frac{e}{d}$$

A queda de pressao adimensionalizada $Eu$ depende do numero de Reynolds $Re$, da razao geometrica $l/d$ e da rugosidade relativa $e/d$. Isso e a essencia do diagrama de Moody, amplamente usado em engenharia de fluidos.